# TaskEval

TaskEval evaluates any free text output using karenina's two evaluation primitives: [templates](answer-templates.md) for correctness and [rubrics](rubrics/index.md) for quality.

In [ ]:
# Mock cell: ensures examples execute without live API keys.
# This cell is hidden in rendered documentation.
import sys
from unittest.mock import MagicMock

_storage_mock = MagicMock()
_storage_mock.DBConfig = MagicMock()
_storage_mock.load_benchmark = MagicMock()
_storage_mock.save_benchmark = MagicMock()
for _mod in [
    "karenina.storage",
    "karenina.storage.generated_models",
    "karenina.storage.auto_mapper",
    "karenina.storage.operations",
    "karenina.storage.models",
]:
    sys.modules[_mod] = _storage_mock

from karenina.benchmark.task_eval import TaskEval, TaskEvalResult, StepEval
from karenina.ports.messages import Message, ToolUseContent
from karenina.schemas.entities import BaseAnswer
from karenina.schemas.entities.rubric import LLMRubricTrait, RegexTrait, Rubric
from karenina.schemas.config.models import ModelConfig
from karenina.schemas.verification.config import VerificationConfig
from karenina.schemas.verification import VerificationResult
from karenina.schemas.verification.result_components import (
    VerificationResultMetadata,
    VerificationResultTemplate,
    VerificationResultRubric,
)
from karenina.schemas.verification.model_identity import ModelIdentity
from pydantic import Field
from datetime import datetime

_mock_identity = ModelIdentity(interface="mock", model_name="mock-model")

def _make_vr(qid, verify_result=True, llm_scores=None, regex_scores=None, response=""):
    vr = VerificationResult(
        metadata=VerificationResultMetadata(
            question_id=qid, template_id="t1", completed_without_errors=True,
            question_text="(mock)", answering=_mock_identity, parsing=_mock_identity,
            execution_time=0.05, timestamp=datetime.now().isoformat(), result_id="abcdef0123456789",
        ),
        template=VerificationResultTemplate(
            verify_result=verify_result, template_verification_performed=True,
            raw_llm_response=response or "(trace)",
        ),
    )
    if llm_scores or regex_scores:
        vr.rubric = VerificationResultRubric(
            rubric_evaluation_performed=True,
            llm_trait_scores=llm_scores or {}, regex_trait_scores=regex_scores or {},
        )
    return vr

def _mock_evaluate(self, config, step_id=None, merge_strategy=None):
    def _build_step(questions, logs, rubrics):
        step = StepEval()
        merged_text = " ".join(l.text for l in logs if l.text)
        llm_scores, regex_scores = {}, {}
        if rubrics:
            for r in rubrics:
                for t in r.llm_traits:
                    llm_scores[t.name] = True if t.kind == "boolean" else 4
                for t in r.regex_traits:
                    regex_scores[t.name] = True
        for q in questions:
            qid = q.get("id", q.get("question", "q")[:8]) if isinstance(q, dict) else getattr(q, "id", "q")
            step.verification_results[qid] = [
                _make_vr(qid, verify_result=True, llm_scores=llm_scores or None,
                         regex_scores=regex_scores or None, response=merged_text[:200])
            ]
        if not questions and rubrics:
            step.verification_results["synthetic"] = [
                _make_vr("synthetic", verify_result=True, llm_scores=llm_scores or None,
                         regex_scores=regex_scores or None, response=merged_text[:200])
            ]
        return step
    result = TaskEvalResult(task_id=self.task_id, metadata=self.metadata)
    if step_id:
        logs = self.step_logs.get(step_id, [])
        qs = self.step_questions.get(step_id, self.global_questions)
        rs = self.step_rubrics.get(step_id, self.global_rubrics)
        result.per_step[step_id] = _build_step(qs, logs, rs)
    else:
        if self.global_questions or self.global_rubrics:
            result.global_eval = _build_step(self.global_questions, self.global_logs, self.global_rubrics)
        for sid in self._get_available_step_ids():
            logs = self.step_logs.get(sid, [])
            qs = self.step_questions.get(sid, [])
            rs = self.step_rubrics.get(sid, [])
            if qs or rs:
                result.per_step[sid] = _build_step(
                    qs or self.global_questions, logs, rs or self.global_rubrics
                )
    return result

_orig_add_template = TaskEval.add_template


def _mock_add_template(self, template_class, step_id=None):
    question_dict = {
        "id": template_class.__name__.lower(),
        "question": "",
        "raw_answer": "",
        "answer_template": "mock_template",
    }
    self.add_question(question_dict, step_id=step_id)


TaskEval.add_template = _mock_add_template

TaskEval.evaluate = _mock_evaluate

In [ ]:
from karenina.benchmark.task_eval import TaskEval
from karenina.schemas.entities import BaseAnswer
from karenina.schemas.entities.rubric import LLMRubricTrait, Rubric
from pydantic import Field

## What Is TaskEval?

[Benchmark](questions-and-benchmarks/index.md) is a closed-loop workflow: it defines questions, generates responses through the [verification pipeline](verification-pipeline.md), and evaluates them together. TaskEval is the open-loop counterpart. You supply any free text output and evaluate it using karenina's primitives (templates and rubrics), with no question definition or answer generation required.

Use TaskEval whenever you have text that needs structured evaluation:

- **Agent workflows** that produce multi-step traces you want to score for correctness or quality
- **CI pipelines** that capture outputs from production runs and need post-hoc quality checks
- **Any free text** from external systems, user submissions, or one-off experiments that you want to evaluate with karenina's judge LLM machinery

## Example: Evaluating an External Output

Suppose an external system (an agent, a RAG pipeline, a human annotator) produced the following text about a cancer drug:

> *"The primary pharmacological target of venetoclax is BCL2 (B-cell lymphoma 2), a protein that inhibits apoptosis. Venetoclax binds selectively to BCL2, displacing pro-apoptotic proteins and triggering programmed cell death in cancer cells [1]."*

You want to evaluate this output along two dimensions.

**Correctness.** Did the response identify BCL2 as the primary target? This is a factual claim with a definitive right answer. In karenina, you express this with a [template](answer-templates.md): a structured schema that tells a judge LLM what to extract from the text, paired with a `verify()` method that checks the extracted values against ground truth. Here, the template would have a single boolean field ("identifies BCL2 as the target") and `verify()` would check that it is `True`.

**Quality.** Is the response concise and well structured? Does it cite sources? These are qualitative dimensions that exist independently from whether the right target was named. In karenina, you express these with [rubric](rubrics/index.md) traits: each trait describes one quality dimension, and a judge LLM scores the response against it.

TaskEval brings these two primitives together in three steps: **log** the text, **attach** templates and rubrics, **evaluate**. Karenina's judge LLM parses the text into the template, runs `verify()`, and scores each rubric trait. No questions to define, no answers to generate.

See the [workflow page](../workflows/task-eval/index.md) for the complete code walkthrough.

## Benchmark vs TaskEval

| Dimension | Benchmark | TaskEval |
|-----------|-----------|----------|
| Workflow | Closed-loop: generate + evaluate | Open-loop: evaluate only |
| Starting point | Questions (define what to ask) | Traces (record what happened) |
| Answer generation | Pipeline generates responses | You supply pre-recorded outputs |
| Question required | Yes | No |
| Best for | Controlled model comparison | Evaluating any free text output |

## Object Structure

A TaskEval instance holds two parallel scopes: global and per-step. Each scope stores logs (the text to evaluate), templates (correctness criteria), and rubrics (quality criteria).

```
┌───────────────────────────────────────────────────────┐
│                      TaskEval                         │
│                                                       │
│  task_id: "agent-drug-lookup"                         │
│  metadata: {"model": "claude-sonnet-4"}               │
│  merge_strategy: "concatenate"                        │
│                                                       │
│  ┌─────────────────────────────────────────────────┐  │
│  │ Global Scope                                    │  │
│  │                                                 │  │
│  │  logs        ← log(), log_trace()               │  │
│  │  templates   ← add_template(Answer)             │  │
│  │  rubrics     ← add_rubric(Rubric)               │  │
│  └─────────────────────────────────────────────────┘  │
│                                                       │
│  ┌──────────────────┐  ┌──────────────────┐           │
│  │ Step: retrieval  │  │ Step: synthesis  │  ...      │
│  │                  │  │                  │           │
│  │  logs            │  │  logs            │           │
│  │  templates       │  │  templates       │           │
│  │  rubrics         │  │  rubrics         │           │
│  └──────────────────┘  └──────────────────┘           │
└───────────────────────────────────────────────────────┘
```

The global scope evaluates all logs together. Per-step scopes evaluate only the logs, templates, and rubrics attached to that step. When you call `evaluate()` without a `step_id`, TaskEval runs global evaluation first, then auto-evaluates each step that has data.

## Attaching Evaluation Criteria

TaskEval accepts two types of evaluation criteria: [answer templates](answer-templates.md) for correctness and [rubrics](rubrics/index.md) for quality. The [evaluation mode](evaluation-modes.md) is auto-detected based on what you attach.

### Templates (correctness)

Pass a `BaseAnswer` subclass directly via `add_template()`. No question text is needed; the template's fields and `verify()` method define what the judge LLM extracts and how correctness is determined.

In [ ]:
from karenina.schemas.entities import BaseAnswer
from pydantic import Field

class Answer(BaseAnswer):
    identifies_bcl2: bool = Field(
        description="True if the response identifies BCL2 as the primary target of venetoclax."
    )
    def ground_truth(self):
        self.correct = {"identifies_bcl2": True}
    def verify(self) -> bool:
        return self.identifies_bcl2 == self.correct["identifies_bcl2"]

task = TaskEval(task_id="my-eval")
task.log("BCL2 is the primary pharmacological target of venetoclax.")
task.add_template(Answer)

### Rubrics (quality)

Attach a `Rubric` via `add_rubric()`. Rubric evaluation uses the same infrastructure as Benchmark: LLM traits, regex traits, callable traits, and metric traits. All Benchmark evaluation options apply, including batch vs sequential strategy and full-trace vs last-message scope.

In [ ]:
task = TaskEval(task_id="my-eval")
task.log("BCL2 is the primary pharmacological target of venetoclax.")
task.add_rubric(Rubric(llm_traits=[
    LLMRubricTrait(name="conciseness", kind="boolean", description="True if the response is concise and direct.")
]))

### Both together

Attach a template and rubric to the same TaskEval for combined correctness and quality evaluation:

In [ ]:
task = TaskEval(task_id="my-eval")
task.log("BCL2 is the primary pharmacological target of venetoclax.")
task.add_template(Answer)
task.add_rubric(Rubric(llm_traits=[
    LLMRubricTrait(name="conciseness", kind="boolean", description="True if the response is concise and direct.")
]))

## Logging

TaskEval provides two logging methods for recording agent outputs.

### Text Logging

Use `log()` for plain text outputs. The text is treated as potential answer content for evaluation.

In [ ]:
task = TaskEval(task_id="my-eval")
task.log("The primary target of venetoclax is BCL2.")
task.log("Additional context about the drug mechanism.", step_id="reasoning")

### Trace Logging

Use `log_trace()` for structured `Message` traces from `karenina.ports.messages`. This preserves the full conversation structure, including tool calls and results.

In [ ]:
from karenina.ports.messages import Message, ToolUseContent

task = TaskEval(task_id="my-eval")
task.log_trace([
    Message.user("What is the target of venetoclax?"),
    Message.assistant("Let me look that up.", tool_calls=[
        ToolUseContent(id="call_1", name="search_db", input={"query": "venetoclax target"})
    ]),
    Message.tool_result(tool_use_id="call_1", content="BCL2 (B-cell lymphoma 2)"),
    Message.assistant("The primary pharmacological target of venetoclax is BCL2."),
])

### Scoping Parameters

Both methods accept `step_id` and `target` parameters:

- **`step_id`**: Groups logs under a named step for per-phase evaluation
- **`target`**: Controls where logs are stored: `"global"`, `"step"`, or `"both"` (default). When `target="both"` and a `step_id` is provided, the log is stored in both the global and step-specific stores.

See the [workflow page](../workflows/task-eval/index.md) for full examples.

## Merge Strategies

When evaluation runs, TaskEval combines logged outputs into a single response for the pipeline. Two strategies control this merging:

**`concatenate`** (default): Combines text and trace logs into a single response. Text logs are wrapped as `Message.assistant()` objects; trace messages are included as-is. All messages are serialized together.

**`traces_only`**: Uses only logs that contain structured `Message` traces. Text-only logs are ignored. Use this when your text logs are debugging output that should not be part of the evaluated response.

In [ ]:
# Use traces_only when text logs are just debug output
task = TaskEval(task_id="my-eval", merge_strategy="traces_only")
task.log("Debug: starting search")  # ignored during evaluation
task.log_trace([Message.assistant("BCL2 is the target.")])  # used

## Step-Specific Evaluation

TaskEval supports both global and step-level scoping for logs, questions, and rubrics. This enables per-phase analysis of multi-step agent workflows.

In [ ]:
task = TaskEval(task_id="agent-run")

# Log outputs for each phase
task.log("Found 3 relevant papers.", step_id="retrieval")
task.log("BCL2 is the primary target based on the evidence.", step_id="synthesis")

# Attach step-specific rubrics
task.add_rubric(
    Rubric(llm_traits=[
        LLMRubricTrait(name="retrieval_quality", kind="boolean", description="True if relevant sources were found.")
    ]),
    step_id="retrieval",
)
task.add_rubric(
    Rubric(llm_traits=[
        LLMRubricTrait(name="synthesis_quality", kind="boolean", description="True if the synthesis is well-supported.")
    ]),
    step_id="synthesis",
)

When you call `evaluate()` without a `step_id`, TaskEval runs global evaluation and then auto-evaluates all steps that have logs. Results are available in `result.global_eval` and `result.per_step`.

## How TaskEval Uses the Pipeline

TaskEval injects logged outputs as `cached_answer_data` into the [verification pipeline](verification-pipeline.md). This skips answer generation (stage 2) but runs all other relevant stages: template parsing (stage 7), template verification (stage 8), rubric evaluation (stage 11), and finalization (stage 13).

Use `VerificationConfig(parsing_only=True)` since no answering model is needed; only a parsing (judge) model is required.

In [ ]:
from karenina.schemas.config.models import ModelConfig
from karenina.schemas.verification.config import VerificationConfig

config = VerificationConfig(
    parsing_models=[ModelConfig(id="haiku", model_provider="anthropic", model_name="claude-haiku-4-5")],
    parsing_only=True,
)
result = task.evaluate(config)

The pipeline treats logged output exactly like a live LLM response. All template parsing, verification, and rubric evaluation stages operate identically to the standard Benchmark workflow.

<div class="admonition note">
<p class="admonition-title">Pipeline Reuse</p>
<p>TaskEval reuses the same verification pipeline as Benchmark. The only difference is that stage 2 (answer generation) is skipped because the response is already provided via <code>cached_answer_data</code>.</p>
</div>